### Script to get a reward for a conversation using the NVIDIA free API

In [1]:
import json
from tqdm import tqdm
from openai import OpenAI

In [ ]:
import json
from openai import OpenAI

def load_jsonl(jsonl_path):
    """
    Loads a line-delimited JSON (.jsonl) file into a list of dicts.
    """
    data = []
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:  # Skip empty lines
                data.append(json.loads(line))
    return data

def load_json(json_path):
    """
    Loads a standard JSON file (containing a list or dict) into a Python object.
    """
    with open(json_path, 'r', encoding='utf-8') as f:
        return json.load(f)

# 1. Load the .jsonl file
meditron_response_name = "_upto_4995"
meditron_responses = load_jsonl("../results/meditron_responses" + meditron_response_name + ".jsonl")

# 2. Load the .json file
prompts = load_json("../results/parsed_prompts_tasks_x_topics_x_answerstyles.json")

# Initialize your client
client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key="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"
)

prompts_by_id = {item['id']: item for item in prompts}

# This list will hold all final results
results_list = []

for i, item in enumerate(tqdm(meditron_responses)):
    prompt_id = item.get('prompt_id')
    seq_num = item.get('sequence_number')
    response_text = item.get('response')

    match = prompts_by_id.get(prompt_id)
    if match:
        # Call the completion API
        completion = client.chat.completions.create(
            model="nvidia/llama-3.1-nemotron-70b-reward",
            messages=[
                {"role": "user", "content": match["prompt"]},
                {"role": "assistant", "content": response_text}
            ]
        )
        reward = completion.choices[0].message.content
        
        # Build the result dictionary
        # ID is a combination of prompt_id and sequence_number
        result_dict = {
            "id": f"{prompt_id}-{seq_num}",
            "messages": [
                {"role": "user", "content": match["prompt"]},
                {"role": "assistant", "content": response_text}
            ],
            "reward": reward[7:]
        }
        
        # Append it to our results list
        results_list.append(result_dict)

        if i % 500 == 0:
            print("Display Example result:")
            print(result_dict)


# 3. Write all results to a new JSON file
output_path = "../results/DPO_dataset_" + meditron_response_name + ".json"
with open(output_path, "w", encoding="utf-8") as outfile:
    # Use indent=2 for readable formatting; ensure_ascii=False for UTF-8
    json.dump(results_list, outfile, ensure_ascii=False, indent=2)

print(f"Results written to {output_path}")

  0%|          | 1/2470 [00:00<28:10,  1.46it/s]

Display Example result:
{'id': '1996-0', 'messages': [{'role': 'user', 'content': 'I am currently responsible for integrating predictive analytics into patient management strategies in a hospital setting. I am evaluating the case of a 65-year-old male patient with a history of hypertension and type 2 diabetes, who presented with an acute myocardial infarction one week ago. Despite successful intervention and stabilization, I am concerned about the potential risk of future cardiovascular events due to his medical history and socioeconomic factors, including limited access to follow-up care and medication adherence issues due to financial constraints. Can you provide guidance on how predictive analytics can be used to forecast this patient’s risk of readmission or future cardiac events, and suggest strategies to leverage these insights in developing cost-effective management plans that could improve his long-term outcomes and mitigate healthcare costs?'}, {'role': 'assistant', 'content':

 20%|██        | 501/2470 [02:56<10:46,  3.04it/s]

Display Example result:
{'id': '2046-0', 'messages': [{'role': 'user', 'content': "I'm working with a 58-year-old female patient who has a family history of type 2 diabetes and is currently showing prediabetic signs with an HbA1c of 6.2%. She has a BMI of 29, follows a sedentary lifestyle, and frequently consumes a high-sugar and high-fat diet. Additionally, she has slightly elevated blood pressure readings averaging 135/85 mmHg. She does not currently take any medication. Given her risk factors and family history, can predictive analytics be utilized to forecast her likelihood of developing type 2 diabetes and cardiovascular disease in the next five years? Also, what preventive strategies would be most effective in mitigating these risks, considering lifestyle modifications and necessary dietary changes?"}, {'role': 'assistant', 'content': "## Addressing Patient's Risks and Preventive Strategies\n\n### Introduction\n\nThank you for your query. I'm a medical AI assistant trained to pro

 41%|████      | 1001/2470 [05:48<08:32,  2.87it/s]

...

Display Example result:
{'id': '2146-0', 'messages': [{'role': 'user', 'content': "I am a physical therapist at a rehabilitation center in Boston, assisting a 58-year-old female patient recently diagnosed with a bacterial infection. The attending physician has prescribed ciprofloxacin for the infection and ibuprofen for pain management. The patient is also on atorvastatin for cholesterol control. Considering the antimicrobial properties and pharmacodynamics of ciprofloxacin, what are the potential drug interactions between these medications, specifically focusing on their microbiological and pharmacological aspects? Additionally, are there any adjustments or monitoring parameters I should be aware of during the patient's physical therapy sessions? Please provide an expert-level explanation."}, {'role': 'assistant', 'content': "### Introduction\n\nAs a medical AI assistant, my response will be structured and detail-oriented, following appropriate medical guidelines and ethical principle

 81%|████████  | 2001/2470 [11:30<02:43,  2.87it/s]

Display Example result:
{'id': '2196-0', 'messages': [{'role': 'user', 'content': "As a chiropractor, I am managing a 45-year-old patient with chronic lower back pain who has recently been prescribed a combination of medications, including gabapentin for neuropathic pain, fluoxetine for mild depression, and amlodipine for hypertension. The patient has undergone genetic testing, revealing a CYP2D6 poor metabolizer status, and carries the HLA-B*1502 allele. Considering these genetic factors, what potential drug interactions or adverse effects should I be vigilant about, and what adjustments or alternative therapies might be advisable to optimize the patient's treatment regimen while minimizing risk?"}, {'role': 'assistant', 'content': "## Drug Interactions and Adverse Effects in a CYP2D6 Poor Metabolizer with HLA-B*1502 Allele\n\n### Introduction\n\nAs a chiropractor managing a patient with chronic lower back pain, it is crucial to consider potential drug interactions and adverse effects

100%|██████████| 2470/2470 [14:08<00:00,  2.91it/s]

Results written to ../results/DPO_dataset__upto_2242.json


In [18]:
import json

def load_json(json_path):
    """
    Loads a JSON file into a Python object.
    """
    with open(json_path, 'r', encoding='utf-8') as f:
        return json.load(f)

def concatenate_json_files(json_paths, output_path):
    """
    Loads multiple JSON files, concatenates their content, and writes the combined result to a new JSON file.
    """
    concatenated_data = []
    
    for path in json_paths:
        # Load the current JSON file
        data = load_json(path)
        # Extend the concatenated data with the current file's entries
        concatenated_data.extend(data)
    
    # Write the concatenated data to the output file
    with open(output_path, 'w', encoding='utf-8') as outfile:
        json.dump(concatenated_data, outfile, ensure_ascii=False, indent=2)

    print(f"Combined data written to {output_path}")

# Paths to the input JSON files
json_file_paths = [
    "../results/DPO_test_dataset_252-7.json",  # Replace with the actual path to the first JSON file
    "../results/DPO_test_dataset_636-9.json"  # Replace with the actual path to the second JSON file
]

# Path to the output JSON file
output_file_path = "../results/DPO_test_dataset.json"

# Concatenate the JSON files
concatenate_json_files(json_file_paths, output_file_path)


Combined data written to ../results/DPO_test_dataset.json
